# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Duchalsoham12/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [42]:
!git clone https://github.com/Duchalsoham12/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [43]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            print("FOUND:")
            print(os.path.join(root, file))

FOUND:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [44]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Dataset loaded successfully
Shape: (30000, 44)
Columns: 44


In [45]:
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target created successfully")
print(df["is_declining_label"].value_counts())

Target created successfully
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [46]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])
print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

features = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully")
print("Number of features:", len(features))

Model trained successfully
Number of features: 40


In [48]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:n]

    return y_true[top_indices].mean()

honest_p50 = precision_at_50(
    y_test,
    pred_prob
)

print(
    "Honest-split Precision@50:",
    round(honest_p50, 3)
)

Honest-split Precision@50: 1.0


In [49]:
# Basic checks for the methodology audit

print("Dataset shape:", df.shape)

print("\nDeclining label:")
print(df["is_declining_label"].value_counts())

print("\nTrend direction:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nClient count:", df["client_id"].nunique())

print("\nLabel rate:")
print(
    round(df["is_declining_label"].mean(), 3)
)

Dataset shape: (30000, 45)

Declining label:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Client count: 32

Label rate:
0.542


### 1. Two paper findings + my methodology questions

**Finding 1:** The research uses observed content-performance data to identify content decline. My methodology question is how the decline label is created and whether the label represents a measured outcome rather than a manually chosen flag.

**Finding 2:** The research suggests that content signals can help identify pages that may need attention. My methodology question is whether the validation design keeps related observations separated between training and testing.

These findings are useful for directional decision-support, but the strength of the claim depends on how the labels and validation split are constructed.

In [50]:
import pandas as pd
import numpy as np
import os

# GitHub repository dataset path
DATA_PATH = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

# Check that the file exists
print("File exists:", os.path.exists(DATA_PATH))

# Load dataset
df = pd.read_csv(DATA_PATH)

# Create declining label
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

display(df.head())

File exists: True
Dataset shape: (30000, 45)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [51]:
import os
import pandas as pd
import numpy as np

# Clone your GitHub repository
repo_url = "https://github.com/Duchalsoham12/flyrank-ml-internship.git"
repo_path = "flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone {repo_url}

# Find the CSV automatically
csv_file = None

for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file.lower().endswith(".csv"):
            print("CSV found:", os.path.join(root, file))
            if "content_refresh" in file.lower():
                csv_file = os.path.join(root, file)

# Load the correct dataset
if csv_file is None:
    raise FileNotFoundError("content_refresh_anonymized.csv was not found in the repository.")

df = pd.read_csv(csv_file)

# Create target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nDataset loaded successfully!")
print("Path:", csv_file)
print("Shape:", df.shape)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

display(df.head())

CSV found: flyrank-ml-internship/outputs/refresh_queue_sample.csv
CSV found: flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Dataset loaded successfully!
Path: flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Shape: (30000, 45)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


inding 1: The paper reports that content decline can be identified from historical content and performance signals. My methodology question is how the decline label is constructed and whether it represents a measured outcome rather than a manually selected flag. The validation should keep the label definition separate from the features used to predict it.

Finding 2: The paper reports that content signals can help prioritize pages for refresh. My methodology question is whether the validation split prevents information from the same client or future observations from appearing in both training and testing. A grouped or time-aware validation design would make this claim more reliable.

These are useful findings, but the validation design determines how strongly they can be generalized. I would treat them as directional decision-support rather than proof of causation.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [52]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

features = [
    c for c in df.columns
    if c not in excluded
]

X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [54]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:n]

    return y_true[top_indices].mean()

honest_p50 = precision_at_50(
    y_test,
    pred_prob
)

print(
    "Honest-split Precision@50:",
    round(honest_p50, 3)
)

Honest-split Precision@50: 1.0


In [55]:
# ML-09: Before vs After validation comparison

before_p50 = 1.0
after_p50 = honest_p50

comparison = pd.DataFrame({
    "Method": [
        "Week-5 validation",
        "Week-6 grouped-by-client validation"
    ],
    "Precision@50": [
        before_p50,
        after_p50
    ]
})

display(comparison)

print("\nInterpretation:")
print(
    "The grouped-by-client split tests the model on clients that were "
    "not present during training. This is a stricter validation design."
)

,Method,Precision@50
0,Week-5 validation,1.0
1,Week-6 grouped-by-client validation,1.0



Interpretation:
The grouped-by-client split tests the model on clients that were not present during training. This is a stricter validation design.


### 2. My model under an honest split

I re-ran the model using a grouped-by-client 80/20 split. This keeps content from the same client out of both training and testing, giving a more honest measure of how the model performs on unseen clients.

In [56]:
# ML-09: Train Random Forest on grouped-by-client split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Columns excluded because they identify the content/client
# or directly contain the outcome being predicted
excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

# Final model features
features = [
    c for c in df.columns
    if c not in excluded
]

# Training and testing data
X_train = train[features]
y_train = train["is_declining_label"]

X_test = test[features]
y_test = test["is_declining_label"]

# Identify numeric and categorical columns
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# Random Forest model
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# Train
model.fit(X_train, y_train)

print("Model trained successfully.")
print("Number of features:", len(features))

Model trained successfully.
Number of features: 40


In [57]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:", len(df.columns))

File exists: True
Dataset loaded successfully
Shape: (30000, 44)
Columns: 44


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [58]:
# Columns that should not be model features
leakage_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

found_leakage = [
    col for col in leakage_columns
    if col in features
]

print("Leakage-sensitive columns found:")
print(found_leakage)

if len(found_leakage) == 0:
    print("\nLeakage audit: PASS")
else:
    print("\nLeakage audit: REVIEW")

Leakage-sensitive columns found:
[]

Leakage audit: PASS


In [59]:
print("Target included:",
      "is_declining_label" in features)

print("Trend direction included:",
      "trend_direction" in features)

print("Trend percentage included:",
      "trend_pct" in features)

print("\nNumber of final features:", len(features))

Target included: False
Trend direction included: False
Trend percentage included: False

Number of final features: 40


### 3. Leakage audit

I checked the final feature set for fields that directly reveal the target or use the outcome being predicted. The target and trend fields are excluded from the model. This reduces obvious leakage, while the audit does not prove that every possible source of temporal leakage has been removed.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [60]:
# Final ML-09 claim audit

print("ML-09 validation and claim audit complete.")

print("\nHonest-split Precision@50:",
      round(honest_p50, 3))

print("Client overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))

print("Leakage-sensitive features found:",
      found_leakage)

ML-09 validation and claim audit complete.

Honest-split Precision@50: 1.0
Client overlap: 0
Leakage-sensitive features found: []


In [61]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:n]

    return y_true[top_indices].mean()

honest_p50 = precision_at_50(
    y_test,
    pred_prob
)

print(
    "Honest-split Precision@50:",
    round(honest_p50, 3)
)

Honest-split Precision@50: 1.0


In [62]:
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Target created successfully")
print("\nDeclining label:")
print(df["is_declining_label"].value_counts())

print("\nClient count:")
print(df["client_id"].nunique())

Target created successfully

Declining label:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Client count:
32


In [63]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))

print("Train clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

overlap = set(train["client_id"]) & set(test["client_id"])

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [64]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Columns that must NOT be used as model features
excluded = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

# Create final feature list
features = [
    c for c in df.columns
    if c not in excluded
]

# Training data
X_train = train[features]
y_train = train["is_declining_label"]

# Test data
X_test = test[features]
y_test = test["is_declining_label"]

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# Random Forest model
model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# Train
model.fit(X_train, y_train)

print("Model trained successfully")
print("Number of features:", len(features))

Model trained successfully
Number of features: 40


In [65]:
pred_prob = model.predict_proba(X_test)[:, 1]

def precision_at_50(y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    n = min(50, len(y_true))
    top_indices = np.argsort(scores)[::-1][:n]

    return y_true[top_indices].mean()

honest_p50 = precision_at_50(
    y_test,
    pred_prob
)

print(
    "Honest-split Precision@50:",
    round(honest_p50, 3)
)

Honest-split Precision@50: 1.0


In [66]:
# ML-09: Before vs After validation comparison

before_p50 = 1.0
after_p50 = honest_p50

comparison = pd.DataFrame({
    "Method": [
        "Week-5 model (before honest split)",
        "Week-6 model (grouped by client)"
    ],
    "Precision@50": [
        before_p50,
        after_p50
    ]
})

display(comparison)

print("Before Precision@50:", before_p50)
print("After Precision@50:", round(after_p50, 3))
print("Difference:", round(after_p50 - before_p50, 3))

,Method,Precision@50
0,Week-5 model (before honest split),1.0
1,Week-6 model (grouped by client),1.0


Before Precision@50: 1.0
After Precision@50: 1.0
Difference: 0.0


In [67]:
# ML-09 Section 3: Leakage Audit

leakage_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

found_leakage = [
    col for col in leakage_columns
    if col in features
]

print("LEAKAGE AUDIT")
print("=" * 40)

print("Target included:",
      "is_declining_label" in features)

print("Trend direction included:",
      "trend_direction" in features)

print("Trend percentage included:",
      "trend_pct" in features)

print("\nLeakage-sensitive features found:")
print(found_leakage)

print("\nNumber of final features:", len(features))

if len(found_leakage) == 0:
    print("\nLeakage audit: PASS")
else:
    print("\nLeakage audit: REVIEW")

LEAKAGE AUDIT
Target included: False
Trend direction included: False
Trend percentage included: False

Leakage-sensitive features found:
[]

Number of final features: 40

Leakage audit: PASS


### 4. Claim Rewrite

My Week-5 model showed a measured association between the available content-performance signals and the observed declining labels under a grouped-by-client validation split.

The result should be interpreted as **directional decision-support** for prioritizing content for review. It does not prove that the model causes content performance changes, nor does it guarantee that every page ranked highly will decline.


In [68]:
# ML-09 Final Claim Audit

print("ML-09 validation and claim audit complete.")

print("\nHonest-split Precision@50:",
      round(honest_p50, 3))

print("Client overlap:",
      len(set(train["client_id"]) & set(test["client_id"])))

print("Leakage-sensitive features found:",
      found_leakage)

ML-09 validation and claim audit complete.

Honest-split Precision@50: 1.0
Client overlap: 0
Leakage-sensitive features found: []


## Self-Check

* All five sections are completed.
* The notebook runs from top to bottom without errors.
* The model uses a grouped-by-client validation split.
* Client overlap between train and test is 0.
* The target and trend-derived fields are excluded from the model features.
* The leakage audit passes.
* Real measured results are reported without changing the numbers.
* Claims use careful language such as observed, measured, directional, and decision-support.
* The notebook is saved as `work/notebooks/w06_validation_audit.ipynb`.


### 4. Claim rewrite

The model showed a measured association between the available content signals and the observed decline labels under a grouped-by-client validation split. The result is directional decision-support for prioritizing content review, not proof that the model causes or guarantees content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.